### This code populates BD_RiesgoClimatico_IKI

\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import sqlite3
import os
from openpyxl import load_workbook
import re

### Populating table WaScenarios

In [2]:
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [3]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

wa_scenarios_df = pd.read_sql_query(
    "SELECT * FROM Scenarios;",
    conn_wa
)

conn_wa.close()

wa_scenarios_df = wa_scenarios_df.rename(
    columns={"ScnID": "WaScnID"}
)

wa_scenarios_df


,WaScnID,Scenario
0,1,CC_CMIP6_85_2050
1,3,Linea_Base_2020
2,4,Linea_Base_2020_Embalses


In [14]:
# WaterALLOC scenario info: WaScnID -> (WaScnName, Description)
wa_scenario_info = {
    3: ("Linea_Base_2020", "Baseline hydrology (1981–2020, no additional infrastructure)"),
    4: ("Linea_Base_2020_Embalses", "Baseline hydrology (1981–2020, includes reservoirs)"),
    1: ("CC_CMIP6_85_2050", "Future climate projection (CMIP6, SSP5-8.5, 2050)"),
    # add more as needed, refer to the printed output from the cell above for list of scenarios
}

In [15]:
# add mapping to scenario id
def normalize_text(s):
    if s is None:
        return ""
    s = s.upper()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# adjust/add as we add more scenarios

def assign_scnid_from_text(scenario_name):
    s = normalize_text(scenario_name)

    if "LINEA BASE" in s:
        return 1
    if "CC CMIP6 85" in s:
        return 2

    return None

In [16]:
wa_scenarios_df["ScnID"] = wa_scenarios_df["Scenario"].apply(assign_scnid_from_text)

In [17]:
wa_desc_lookup = {
    wa_scn_id: desc
    for wa_scn_id, (_, desc) in wa_scenario_info.items()
}

wa_scenarios_df["Description"] = wa_scenarios_df["WaScnID"].map(wa_desc_lookup)


In [18]:
wa_scenarios_df

,WaScnID,Scenario,ScnID,Description
0,1,CC_CMIP6_85_2050,2,"Future climate projection (CMIP6, SSP5-8.5, 2050)"
1,3,Linea_Base_2020,1,"Baseline hydrology (1981–2020, no additional i..."
2,4,Linea_Base_2020_Embalses,1,"Baseline hydrology (1981–2020, includes reserv..."


In [ ]:
# Insert into WaScenarios table in the SQL DB
rows_to_insert = list(
    wa_scenarios_df[["WaScnID", "Scenario", "Description", "ScnID"]]
    .itertuples(index=False, name=None)
)

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS WaScenarios;")
    cursor.execute("""
        CREATE TABLE WaScenarios (
            WaScnID INTEGER NOT NULL UNIQUE,
            WaScnName TEXT NOT NULL,
            Description TEXT, 
            ScnID INTEGER
        );
    """)

    cursor.executemany(
        "INSERT INTO WaScenarios (WaScnID, WaScnName, Description, ScnID) VALUES (?, ?, ?, ?);",
        rows_to_insert
    )

    conn.commit()

### Populating table ScnMod

In [20]:
# Connect to the SQLite database
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [ ]:
scenarios_data = [
    (1, 'Linea Base', 'Baseline scenario (1981-2020)'),
    (2, 'CC CMIP6 85', 'Future scenario 2050'),
    # add more as needed
]

In [ ]:
# Write the ScnMod table to the database
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS ScnMod;")
    cursor.execute("""
        CREATE TABLE ScnMod (
            ScnID INTEGER PRIMARY KEY,
            ScnName TEXT NOT NULL,
            Description TEXT,
        );
    """)

    cursor.executemany(
        "INSERT INTO ScnMod (ScnID, ScnName, Description) VALUES (?, ?, ?);",
        scenarios_data
    )

    conn.commit()

print("ScnMod table populated successfully.")

ScnMod table populated successfully.


### Create table for the WaterALLOC Indicators

In [ ]:
# Connect to the SQLite database
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [24]:
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS IndValues_WaALLOC;")
    cursor.execute("""
        CREATE TABLE IndValues_WaALLOC (
            WaScnID INTEGER,
            IndID INTEGER,
            COMID INTEGER, 
            Value REAL,
            PRIMARY KEY (WaScnID, IndID, COMID)
        );
    """)

    conn.commit()

print("IndValues_WaALLOC table created successfully.")

IndValues_WaALLOC table created successfully.


### Create Indicators Table

In [25]:
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"

In [27]:
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS Indicators;")
    cursor.execute("""
        CREATE TABLE Indicators (
            IndID INTEGER NOT NULL,
            CompID INTEGER NOT NULL,
            Name TEXT NOT NULL UNIQUE,
            TextID TEXT NOT NULL UNIQUE,
            Sector NUMERIC,
            Factor TEXT,
            Type TEXT,
            SIG_Type TEXT,
            Active TEXT,
            "Order" TEXT,
            Max NUMERIC,
            Min NUMERIC,
            IndTable TEXT
        );
    """)

    conn.commit()

print("Indicators table created successfully.")

Indicators table created successfully.


### Populating the indicators table for hazards (P1 to P13)

In [28]:
# Connect to your database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()


# BUFFER MAX by 5% for indicators that are not categorical 
# add another column to indicate which table each indicator should populate into
data = [
    (101, 1, 'SPI intensity and frequency (Standarized Precipitation Index)', 'P1', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 4, 1, 'Dynamic'),
    (102, 1, 'Rainfall intensity', 'P2', 'All', 'Hazard', 'Optimum', 'Python', 'True', 'ASC', 4, 1, 'Dynamic'), # interquartile rank
    (103, 1, 'Erosivity', 'P3', 'All', 'Hazard', 'Optimum', 'Python', 'True', 'ASC', 24500, 0, 'Dynamic'), # selected min/max based on all the COMIDS + a buffer for the max value
    (104, 1, 'Change in mean annual temperature', 'P4', 'All', 'Hazard', 'Indirect', 'Python', 'True', 'ASC', 4, 1, 'Dynamic'), # asocian los 4 niveles de peligro a rangos interquantiles
    (105, 1, 'Glacial retreat rate or index', 'P5', 'All', 'Hazard', 'Indirect', 'Python/GIS', 'True', 'ASC', 51, 0, 'Dynamic'), # solo en comids con masa glaciares
    (106, 1, 'Susceptibility to landslides', 'P6', 'All', 'Hazard', 'Optimum', 'Python/GIS', 'True', 'ASC', 4, 1, 'Static'), # Categorias muy baja, baja, media, alta, muy alta
    (107, 1, 'Susceptibility to fluvial flooding', 'P7', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 4, 1,'Static'),
    (108, 1, 'Soil susceptibility to water erosion', 'P8', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 1342, 0, 'Dynamic'),
    (110, 1, 'Volume of the glacial lagoon', 'P10', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 1, 0, 'Dynamic'),
    (111, 1, 'Slope of the glacier mass', 'P11', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 1, 0, 'Dynamic'),
    (112, 1, 'Distance from the glacier to the lagoon', 'P12', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 1, 0, 'Dynamic'),
    (113, 1, 'Water yield: Water production per basin and per square kilometer', 'P13', 'All', 'Hazard', 'Optimum', 'Python/WaterAlloc', 'True', 'ASC', 1486, 100, 'WaterAlloc'),
]

# Adjusted INSERT query to match the new order
insert_query = """
INSERT OR REPLACE INTO Indicators (
    IndID, CompID, Name, TextID, Sector, Factor, Type, SIG_type, Active, "Order", Max, Min, IndTable
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""
cursor.execute("DELETE FROM Indicators WHERE TextID LIKE 'P%';")
conn.commit()

cursor.executemany(insert_query, data)
conn.commit()
conn.close()

### Populating the indicators table for Exposure (E1 to E8)

In [29]:
# Connect to your database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Updated data to match the new column order; min for most Exposicion indicators should be 1, because 0s will be replaced with NaN
data = [
    (201, 2, 'Number of agricultural producers', 'E1', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 5200, 1, 'Static'),
    (202, 2, 'Number of irrigation water infrastructure exposed to climate hazards', 'E2', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 2250, 1, 'Static'),
    (203, 2, 'Number of hectares of agricultural land, natural pastures and forested land', 'E3', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 85000, 1, 'Static'),
    (204, 2, 'Number of heads of livestock production', 'E4', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 104000, 1, 'Static'),
    (206, 2, 'Number of people with water supply for population purposes', 'E6', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 2200000, 1, 'Static'),
    (207, 2, 'Number of infrastructure works for the provision of drinking water exposed to climate hazards', 'E7', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 160, 1, 'Static'),
    (208, 2, 'Number of homes with electric lighting by public network', 'E8', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 630000, 1, 'Static'),
    (209, 2, 'Number of hydro-energy and electrical infrastructure works exposed to climate hazards', 'E9', 'Demographic', 'Exposure', 'Optimum', 'WaterAlloc', 'True', 'ASC', 10, 1, 'Dynamic'),
]

# Adjusted INSERT query to match the new order
insert_query = """
INSERT OR REPLACE INTO Indicators (
    IndID, CompID, Name, TextID, Sector, Factor, Type, SIG_type, Active, "Order", Max, Min, IndTable
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

cursor.execute("DELETE FROM Indicators WHERE TextID LIKE 'E%';")
conn.commit()

cursor.executemany(insert_query, data)
conn.commit()
conn.close()

### Populating the indicators table for vulnerability VSB (VSB1 to VSB8), VSS (VSS2 to VSS8), VCA (CA1 to VCA15) (27 in total)

In [31]:
# Connect to your database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Updated data to match the new column order
#ADD additional vulnerability indicators
data = [
    (301, 3, 'Index of the volume of water granted for agricultural purposes', 'VSB1', 'Agriculture', 'Vulnerability', 'Indirect', 'Wateralloc', 'True', 'DESC', 16300000, 0, 'Static'),
    (302, 3, 'Environmental Quality Index of Surface Water Resources - ICARHS (Cat 3: Irrigation and drinking  water for animals )', 'VSB2', 'Agriculture', 'Vulnerability', 'Optimum', 'ANA', 'True', 'DESC', 1, 0, 'Static'),
    (303, 3, 'ICARHS (Cat 1: Demographic and recreational)', 'VSB3', 'Demographic', 'Vulnerability', 'Optimum', 'ANA', 'True', 'DESC', 1, 0, 'Static'),
    (304, 3, 'Total Susepended Solids Index - ICARHS (Cat 4: Conservation of the aquatic environment, rivers of the coast, highlands, and jungle)', 'VSB4', 'Energy', 'Vulnerability', 'Indirect', 'ANA', 'True', 'DESC', 1, 0, 'Static'),
    (306, 3, 'Theoretical hydroelectric potential', 'VSB6', 'Energy', 'Vulnerability', 'Indirect', 'Wateralloc', 'True', 'ASC', 47000, 0, 'WaterALLOC'),
    (308, 3, 'Priority ecosystem degradation index', 'VSB8', 'All', 'Vulnerability', 'Optimum', 'GIS', 'True', 'ASC', 1, 0, 'Static'),
    (310, 3, 'Availability of water by basin for the agricultural sector', 'VSB10', 'All', 'Vulnerability', 'Optimum', 'WaterALLOC', 'True', 'DESC', 1, 0, 'WaterALLOC'), # update min/max and order 
]

# Adjusted INSERT query to match the new order
insert_query = """
INSERT OR REPLACE INTO Indicators (
    IndID, CompID, Name, TextID, Sector, Factor, Type, SIG_type, Active, "Order", Max, Min, IndTable
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""
cursor.execute("DELETE FROM Indicators WHERE TextID LIKE 'VSB%';")
conn.commit()

cursor.executemany(insert_query, data)
conn.commit()
conn.close()

In [32]:
# Connect to your database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# if highest risk is 1 then ascending if highest risk is 0 then descending
data = [
    (401, 3, 'Index of mechanized/modern irrigation for agricultural production', 'VSS1', 'Agriculture', 'Vulnerability', 'Indirect', 'GIS', 'True', 'ASC', 1, 0, 'Static'),
    (402, 3, 'Percentage of irrigation water fee collection', 'VSS2', 'Agriculture', 'Vulnerability', 'Indirect', 'Wateralloc', 'True', 'ASC', 1, 0, 'Static'),
    (403, 3, 'Index of formalized irrigation water use rights', 'VSS3', 'Agriculture', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0,'Static'),
    (404, 3, 'Index of human development', 'VSS4', 'Agriculture', 'Vulnerability', 'Optimum', 'GIS', 'True', 'ASC', 1, 0, 'Static'),
    (405, 3, 'Index of women participiation in agricultural activities', 'VSS5', 'Agriculture', 'Vulnerability', 'Indirect', 'GIS', 'True', 'ASC', 1, 0, 'Static'),
    (406, 3, 'Volume of annual water demand for demographic use', 'VSS6', 'Demographic', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 45700, 0, 'WaterALLOC'),
    (407, 3, 'Index of annual electricity consumption', 'VSS7', 'Energy', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 182000, 0, 'Static'),
    (408, 3, 'Establishment of the Water Resources Management Council in river basins', 'VSS8', 'All', 'Vulnerability', 'Optimum', 'ANA', 'True', 'DESC', 1, 0, 'Static'),
    (409, 3, 'Supply reliability: Portion of time during which the water demand is fully met for the agricultural sector', 'VSS9', 'Agriculture', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0, 'WaterALLOC'),
    (410, 3, 'Supply reliability: Portion of time during which the water demand is fully met for the municipal sector', 'VSS10', 'Demographic', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0, 'WaterALLOC'), # check min/max and order 
    (411, 3, 'Average deficit, Average magnitude of the supply deficit in the agricultural sector', 'VSS11', 'Agriculture', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0, 'WaterALLOC'), # check min/max and order 
    (412, 3, 'Average deficit, Average magnitude of the supply deficit in the municipal sector', 'VSS12', 'Demographic', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0, 'WaterALLOC'), # check min/max and order
    (4131, 3, 'Index of hydrological stress for the agriculture sector', 'VSS13_A', 'Agriculture', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0, 'WaterALLOC'), # check min/max and order
    (4132, 3, 'Index of hydrological stress for the demographic sector', 'VSS13_P', 'Demographic', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'ASC', 1, 0, 'WaterALLOC'), # check min/max and order
]

# Adjusted INSERT query to match the new order
insert_query = """
INSERT INTO Indicators (
    IndID, CompID, Name, TextID, Sector, Factor, Type, SIG_type, Active, "Order", Max, Min, IndTable
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

cursor.execute("DELETE FROM Indicators WHERE TextID LIKE 'VSS%';")
conn.commit()

cursor.executemany(insert_query, data)
conn.commit()
conn.close()

In [34]:
# Connect to your database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

data = [
    (501, 3, 'Index of agricultural producers with training and technical assistance', 'VCA1', 'Agriculture', 'Vulnerability', 'Indirect', 'GIS', 'True', 'DESC', 1600, 1, 'Static'),
    (503, 3, 'Number of hydraulic and natural infrastructure works for water storage and distribution', 'VCA3', 'Agriculture', 'Vulnerability', 'Optimum', 'Wateralloc', 'True', 'DESC', 1500, 0, 'Static'),
    (504, 3, 'Level of progress of the Climate Change Mitigation and Adaptation Plan (PMACC)', 'VCA4', 'Demographic', 'Vulnerability', 'Indirect', 'ANA', 'True', 'DESC', 1, 0, 'Static'),
    (506, 3, 'Percentage variation rate of the Planta Factor', 'VCA6', 'Energy', 'Vulnerability', 'Indirect', 'Wateralloc', 'True', 'DESC', 1, 0, 'WaterALLOC'),
    (510, 3, 'Progress in the preparation of the Water Resources Management Plan for the basin', 'VCA10', 'All', 'Vulnerability', 'Indirect', 'ANA', 'True', 'DESC', 1, 0, 'Static'),
    (512, 3, 'Density index of hydrometeorological monitoring network stations in the basin', 'VCA12', 'All', 'Vulnerability', 'Indirect', 'GIS', 'True', 'DESC', 1, 0, 'Static'),
    (515, 3, 'Percentage of critical points on riverbanks or streams protected against hazards', 'VCA15', 'All', 'Vulnerability', 'Optimum', 'GIS', 'True', 'ASC', 8.4, 0, 'Static'),
    (5161, 3, 'Relative frequency of recovery after failure for the agricultural sector', 'VCA16_A', 'Agriculture', 'Vulnerability', 'Optimum', 'Python script', 'True', 'DESC', 1, 0, 'WaterALLOC'), # check min/max and order
    (5162, 3, 'Relative frequency of recovery after failure for the demographic sector', 'VCA16_P', 'Demographic', 'Vulnerability', 'Optimum', 'Python script', 'True', 'DESC', 1, 0, 'WaterALLOC'), # check min/max and order
    (5171, 3, 'Percentage of net imported water volume relative to the total demand for the agricultural sector', 'VCA17_A', 'Agriculture', 'Vulnerability', 'Optimum', 'Python script', 'True', 'DESC', 1, 0, 'WaterALLOC'), # check min/max and order
    (5172, 3, 'Percentage of net imported water volume relative to the total demand for the demographic sector', 'VCA17_P', 'Demographic', 'Vulnerability', 'Optimum', 'Python script', 'True', 'DESC', 1, 0, 'WaterALLOC'), # check min/max and order
    (5173, 3, 'Percentage of net imported water volume relative to the total demand for the energy sector', 'VCA17_E', 'Energy', 'Vulnerability', 'Optimum', 'Python script', 'True', 'DESC', 1, 0, 'WaterALLOC'), # check min/max and order
    (518, 3, 'Index of volume of reservoir storage', 'VCA18', 'All', 'Vulnerability', 'Optimum', 'Python script', 'True', 'DESC', 1, 0, 'WaterALLOC'), # check min/max and order 
]

# Adjusted INSERT query to match the new order
insert_query = """
INSERT INTO Indicators (
    IndID, CompID, Name, TextID, Sector, Factor, Type, SIG_type, Active, "Order", Max, Min, IndTable
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,?);
"""

cursor.execute("DELETE FROM Indicators WHERE TextID LIKE 'VCA%';")
conn.commit()

cursor.executemany(insert_query, data)
conn.commit()
conn.close()

### Create Users Table

In [5]:
# Connect to the database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Create the Users table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Users (
    UserID INTEGER NOT NULL UNIQUE PRIMARY KEY,
    Name TEXT NOT NULL
);
""")

# Insert the row
cursor.execute("""
INSERT OR IGNORE INTO Users (UserID, Name)
VALUES (?, ?);
""", (1, 'Sophia'))

# Commit and close
conn.commit()
conn.close()

### Create and Populate Tables for Impact Chains and Impact Chain Indicators

In [3]:
# read in the impact chains excel file or sheet
#pd.read_excel(filepath, sheet_name='Sheet1')
#excelfile_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\Guia_ERC_Indicadores.xlsx'
#impactchains = load_workbook(excelfile_path, data_only=True)
ic_excelfile =  fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Guia_ERC_Indicadores.xlsx"
impactchains = load_workbook(ic_excelfile,data_only=True)
# Select the sheet by name
impactchains = impactchains["Cadenas_Impacto"]

In [4]:
rows = []

for row in impactchains.iter_rows(values_only=True):
    rows.append(list(row))

# Convert to DataFrame
impactchains_df = pd.DataFrame(rows[1:], columns=rows[0])   # first row = header
impactchains_df.head()

,Peligro,Sector,P,E,VSB,VSS,VCA,Risk,None,None,Eliminados,None,Who?,Indicator(s),Question,To do,None
0,Sequias,Energetico,P1,E8,VSB4,VSS7,VCA6,None,Con datos/completo,None,P9,None,None,None,None,None,None
1,None,None,None,None,VSB6,VSS8,None,None,WaterALLOC,None,E5,None,None,None,None,None,None
2,None,None,None,None,VSB8,None,None,None,Esperando datos,None,None,None,None,None,None,None,None
3,None,None,None,None,VSB10_E,None,VCA18,None,Eliminar?,None,VSB5,None,Enrique/Sophia,P1,None,Look into what code is necessary,None
4,Movimientos en masa,Energetico,P2,E9,None,VSS8,VCA15,None,None,None,VSB7,None,Sophia,None,None,"if we do receive future glaciar shapefiles, so...",None


In [5]:
# Rename for desired columns
impactchains_df = impactchains_df.rename(columns={
    "Peligro": "Impact Chain",
    "Sector": "Sector"
})

#Forward fill the grouped impact chain + sector
impactchains_df["Impact Chain"] = impactchains_df["Impact Chain"].ffill()
impactchains_df["Sector"] = impactchains_df["Sector"].ffill()

#Function to collect indicator codes into lists
def collect(series):
    vals = series.dropna().astype(str).tolist()
    return vals if vals else None

#Group and aggregate
impactchains_df_grouped = impactchains_df.groupby(["Impact Chain", "Sector"]).agg({
    "P": collect,
    "E": collect,
    "VSB": collect,
    "VSS": collect,
    "VCA": collect
}).reset_index()

impactchains_df_grouped = impactchains_df_grouped[impactchains_df_grouped["Sector"] != "Indices"]

impactchains_df_grouped

,Impact Chain,Sector,P,E,VSB,VSS,VCA
0,Alteracion de las propiedades fisico-quimicas ...,Agrario,[P4],"[E1, E3]","[VSB1, VSB2]","[VSS2, VSS3, VSS4, VSS5, VSS8]",[VCA1]
1,Alteracion de las propiedades fisico-quimicas ...,Poblacional,[P4],[E6],None,"[VSS6, VSS8]",None
2,Disminucion del caudal,Agrario,"[P4, P5, P13]","[E1, E3, E4]","[VSB1, VSB2, VSB8, VSB10_A]","[VSS1, VSS2, VSS3, VSS4, VSS5, VSS8, VSS9, VSS...","[VCA1, VCA3, VCA16_A, VCA17_A, VCA18]"
3,Disminucion del caudal,Energetico,"[P4, P5, P13]",[E8],"[VSB4, VSB6, VSB8, VSB10_E]","[VSS7, VSS8]","[VCA6, VCA18]"
4,Disminucion del caudal,Poblacional,"[P4, P5, P13]",[E6],"[VSB3, VSB8, VSB10_P]","[VSS6, VSS8, VSS10, VSS12, VSS13_P]","[VCA4, VCA16_P, VCA17_P, VCA18]"
5,Erosion Hidrica,Agrario,"[P3, P8]",[E2],"[VSB2, VSB8]","[VSS2, VSS4, VSS8]","[VCA1, VCA15]"
6,Erosion Hidrica,Energetico,"[P3, P8]",[E9],[VSB8],[VSS8],[VCA15]
7,Erosion Hidrica,Poblacional,"[P3, P8]",[E7],"[VSB3, VSB8]",[VSS8],"[VCA4, VCA10, VCA15]"
8,Inundaciones,Agrario,"[P2, P7]",[E2],None,"[VSS2, VSS4, VSS8]","[VCA1, VCA15]"
9,Inundaciones,Energetico,"[P2, P7]",[E9],None,[VSS8],[VCA15]


In [6]:
# translate to english for importing to DB
impact_chain_map = {
    "Alteracion de las propiedades fisico-quimicas del agua": "Alteration of the Physiochemical Properties of Water",
    "Disminucion del caudal": "Decreased streamflow",
    "Erosion Hidrica": "Water erosion",
    "Inundaciones": "Flooding",
    "Mayor ocurrencia de aluviones": "Increased occurrence of mudslides",
    "Movimientos en masa": "Mass movements",
    "Sequias": "Droughts"
}
sector_map = {
    "Agrario": "Agricultural",
    "Energetico": "Energy",
    "Poblacional": "Demographic",
}

In [7]:
# --- Clean strings ---
impactchains_df_grouped["Impact Chain"] = impactchains_df_grouped["Impact Chain"].str.strip()
impactchains_df_grouped["Sector"] = impactchains_df_grouped["Sector"].str.strip()

# --- Save Spanish originals BEFORE translating ---
impactchains_df_grouped["Impact Chain_ES"] = impactchains_df_grouped["Impact Chain"]
impactchains_df_grouped["Sector_ES"] = impactchains_df_grouped["Sector"]

# --- Translate to English for DB fields ---
impactchains_df_grouped["Impact Chain"] = impactchains_df_grouped["Impact Chain"].replace(impact_chain_map)
impactchains_df_grouped["Sector"] = impactchains_df_grouped["Sector"].replace(sector_map)


In [8]:
impactchains_df_grouped

,Impact Chain,Sector,P,E,VSB,VSS,VCA,Impact Chain_ES,Sector_ES
0,Alteration of the Physiochemical Properties of...,Agricultural,[P4],"[E1, E3]","[VSB1, VSB2]","[VSS2, VSS3, VSS4, VSS5, VSS8]",[VCA1],Alteracion de las propiedades fisico-quimicas ...,Agrario
1,Alteration of the Physiochemical Properties of...,Demographic,[P4],[E6],None,"[VSS6, VSS8]",None,Alteracion de las propiedades fisico-quimicas ...,Poblacional
2,Decreased streamflow,Agricultural,"[P4, P5, P13]","[E1, E3, E4]","[VSB1, VSB2, VSB8, VSB10_A]","[VSS1, VSS2, VSS3, VSS4, VSS5, VSS8, VSS9, VSS...","[VCA1, VCA3, VCA16_A, VCA17_A, VCA18]",Disminucion del caudal,Agrario
3,Decreased streamflow,Energy,"[P4, P5, P13]",[E8],"[VSB4, VSB6, VSB8, VSB10_E]","[VSS7, VSS8]","[VCA6, VCA18]",Disminucion del caudal,Energetico
4,Decreased streamflow,Demographic,"[P4, P5, P13]",[E6],"[VSB3, VSB8, VSB10_P]","[VSS6, VSS8, VSS10, VSS12, VSS13_P]","[VCA4, VCA16_P, VCA17_P, VCA18]",Disminucion del caudal,Poblacional
5,Water erosion,Agricultural,"[P3, P8]",[E2],"[VSB2, VSB8]","[VSS2, VSS4, VSS8]","[VCA1, VCA15]",Erosion Hidrica,Agrario
6,Water erosion,Energy,"[P3, P8]",[E9],[VSB8],[VSS8],[VCA15],Erosion Hidrica,Energetico
7,Water erosion,Demographic,"[P3, P8]",[E7],"[VSB3, VSB8]",[VSS8],"[VCA4, VCA10, VCA15]",Erosion Hidrica,Poblacional
8,Flooding,Agricultural,"[P2, P7]",[E2],None,"[VSS2, VSS4, VSS8]","[VCA1, VCA15]",Inundaciones,Agrario
9,Flooding,Energy,"[P2, P7]",[E9],None,[VSS8],[VCA15],Inundaciones,Energetico


In [9]:
# --- Connect to the database ---
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Drop + recreate table (cleanest way)
cursor.execute("DROP TABLE IF EXISTS ImpactChains;")
cursor.execute("""
CREATE TABLE ImpactChains (
    IcID INTEGER NOT NULL UNIQUE PRIMARY KEY,
    Name TEXT NOT NULL,
    Name_ES TEXT NOT NULL,
    Sector TEXT NOT NULL,
    Sector_ES TEXT NOT NULL
);
""")

# Prepare new rows
impactchain_rows = []
for ic_id, (_, row) in enumerate(impactchains_df_grouped.iterrows(), start=1):
    impactchain_rows.append((
        ic_id,
        row["Impact Chain"],      # Name (EN)
        row["Impact Chain_ES"],   # Name_ES (ES)
        row["Sector"],            # Sector (EN)
        row["Sector_ES"]          # Sector_ES (ES)
    ))

# Insert rows
cursor.executemany("""
INSERT INTO ImpactChains (IcID, Name, Name_ES, Sector, Sector_ES)
VALUES (?, ?, ?, ?, ?);
""", impactchain_rows)

conn.commit()
conn.close()



In [9]:
# Check if indicators in the impact chains excel file are present in the Indicators table
# Connect to database
conn = sqlite3.connect(db_path)

# Load Indicators table
indicators_df = pd.read_sql_query("SELECT TextID FROM Indicators;", conn)
indicator_textids = set(indicators_df["TextID"])

# Collect all TextIDs from impactchains_df_grouped
all_textids = set()

for idx, row in impactchains_df_grouped.iterrows():
    # Combine all indicator columns; handle None
    for col in ["P", "E", "VSB", "VSS", "VCA"]:
        values = row[col]
        if values is not None:
            all_textids.update(values)

# Find any missing TextIDs
missing_textids = all_textids - indicator_textids

if missing_textids:
    print("Warning: The following TextIDs are in impactchains_df_grouped but not in the Indicators table:")
    print(missing_textids)
else:
    print("All TextIDs from impactchains_df_grouped are present in the Indicators table.")

conn.close()

{'VSB10_A', 'VSB10_E', 'VSB10_P'}


In [10]:
# Connect to database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Drop old table if it exists
cursor.execute("DROP TABLE IF EXISTS ImpactChain_Indicators;")

# Create new table
cursor.execute("""
CREATE TABLE ImpactChain_Indicators (
    IcID INTEGER NOT NULL,
    IndID INTEGER NOT NULL,
    TextID TEXT NOT NULL,
    PRIMARY KEY (IcID, IndID)
);
""")

# Fetch all Indicators table into a DataFrame
indicators_df = pd.read_sql_query("SELECT IndID, TextID FROM Indicators;", conn)

# Loop through each Impact Chain in impactchains_df_grouped
for idx, row in impactchains_df_grouped.iterrows():
    ic_id = idx + 1  # IcID starts at 1
    
    # Combine all indicator columns safely
    text_ids = []
    for col in ["P", "E", "VSB", "VSS", "VCA"]:
        values = row[col]
        if isinstance(values, list):
            text_ids.extend(values)
    
    # Keep only the TextIDs that exist in the Indicators table
    matched_indicators = indicators_df[indicators_df["TextID"].isin(text_ids)]
    
    # Prepare rows to insert
    insert_rows = [(ic_id, ind_id, text_id) for ind_id, text_id in zip(matched_indicators["IndID"], matched_indicators["TextID"])]
    
    # Insert into table if there’s anything to insert
    if insert_rows:
        cursor.executemany("""
            INSERT INTO ImpactChain_Indicators (IcID, IndID, TextID)
            VALUES (?, ?, ?);
        """, insert_rows)
        

# Commit and close
conn.commit()
conn.close()

### Create and Populate Tables for Weights 

In [11]:
ic_excelfile =  fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Guia_ERC_Indicadores.xlsx"
impactchains = load_workbook(ic_excelfile,data_only=True)

# Select the wegiths for each type of indicator by sheet by name
pesos_peligro = impactchains["Pesos_Peligro"]
pesos_exposicion = impactchains["Pesos_Exposicion"]
pesos_vulnerabilidad = impactchains["Pesos_Vulnerabilidad"]

In [12]:
def sheet_to_df(sheet):
    """
    Convert an openpyxl worksheet to a pandas DataFrame.
    Assumes first row contains headers.
    """
    rows = [list(row) for row in sheet.iter_rows(values_only=True)]
    return pd.DataFrame(rows[1:], columns=rows[0])


In [13]:
# Apply function to each sheet
pesos_peligro_df = sheet_to_df(pesos_peligro)
pesos_exposicion_df = sheet_to_df(pesos_exposicion)

# Fix header for vulnerabilidad sheet only
rows = [list(r) for r in pesos_vulnerabilidad.iter_rows(values_only=True)]

# Use second row as header, and remaining rows as data
pesos_vulnerabilidad_df = pd.DataFrame(rows[2:], columns=rows[1])



In [14]:
# Standardize column names across peso tables
pesos_exposicion_df = pesos_exposicion_df.rename(
    columns={"Importancia (de nosotros) para el Riesgo": "Importancia"}
)

pesos_vulnerabilidad_df = pesos_vulnerabilidad_df.rename(
    columns={"Importancia (de nosotros) para el Riesgo": "Importancia"}
)

In [15]:
# Clean and standardize peso tables
def clean_pesos(df):
    return df[df["ID"].notna()][["ID", "Factor", "Importancia", "Incertidumbre", "Resolucion","Peso"]]

# Standardize column names across peso tables
pesos_exposicion_df = pesos_exposicion_df.rename(
    columns={"Importancia (de nosotros) para el Riesgo": "Importancia"}
)

pesos_vulnerabilidad_df = pesos_vulnerabilidad_df.rename(
    columns={"Importancia (de nosotros) para el Riesgo": "Importancia"}
)
p_p = clean_pesos(pesos_peligro_df)
p_e = clean_pesos(pesos_exposicion_df)
p_v = clean_pesos(pesos_vulnerabilidad_df)

# Assign Factor labels
p_p = clean_pesos(pesos_peligro_df)
p_p["Factor"] = "P"

p_e = clean_pesos(pesos_exposicion_df)
p_e["Factor"] = "E"

p_v = clean_pesos(pesos_vulnerabilidad_df)
p_v["Factor"] = "V"



In [16]:
pesos_all = pd.concat([p_p, p_e, p_v], ignore_index=True)
pesos_all = pesos_all.dropna(subset=["Peso"])  # drops rows where Peso is NaN

In [17]:
# Base ID for matching
pesos_all["BaseTextID"] = pesos_all["ID"]

In [18]:
# Connect to database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Drop old table if it exists
cursor.execute("DROP TABLE IF EXISTS IndicatorWeights;")

# --- Create Indicator-level Weights table ---
cursor.execute("""
CREATE TABLE IndicatorWeights (
    WeightID INTEGER PRIMARY KEY AUTOINCREMENT,
    UserID TEXT NOT NULL,
    IndID INTEGER NOT NULL,
    Factor TEXT NOT NULL,        -- 'P', 'E', 'V'
    Importance_Subweight INTEGER NOT NULL,
    Uncertainty_Subweight INTEGER NOT NULL,
    Resolution_Subweight INTEGER NOT NULL,
    WeightValue REAL NOT NULL,
    TextID TEXT,
    UNIQUE (UserID, IndID),
    FOREIGN KEY (UserID) REFERENCES Users(UserID),
    FOREIGN KEY (IndID) REFERENCES Indicators(IndID)
);
""")
conn.commit()

# --- Get first UserID ---
cursor.execute("SELECT UserID FROM Users")
user_id = cursor.fetchone()[0]

# --- Load indicators with factor info ---
indicators_df = pd.read_sql_query("""
SELECT IndID, TextID, Factor
FROM Indicators
""", conn)

def base_textid(textid):
    m = re.match(r"^(.*)_(A|E|P)$", textid)
    return m.group(1) if m else textid

indicators_df["BaseTextID"] = indicators_df["TextID"].apply(base_textid)

merged_df = indicators_df.merge(
    pesos_all,
    left_on="BaseTextID",
    right_on="BaseTextID",
    how="inner"
)

# Use Factor from pesos_all (P/E/V)
merged_df['Factor'] = merged_df['Factor_y']


for _, r in merged_df.iterrows():
    cursor.execute("""
        INSERT OR REPLACE INTO IndicatorWeights (
            UserID,
            IndID,
            Factor,
            Importance_Subweight,
            Uncertainty_Subweight,
            Resolution_Subweight,
            WeightValue,
            TextID
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        user_id,
        r['IndID'],
        r['Factor'],
        int(r['Importancia']),
        int(r['Incertidumbre']),
        int(r['Resolucion']),
        float(r['Peso']),
        r['ID']
    ))

conn.commit()
conn.close()

print("IndicatorWeights table populated from pesos_all successfully.")


IndicatorWeights table populated from pesos_all successfully.


In [76]:
# Connect to database
# For now factors will be evenly weighted (1/3)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# Drop old table if it exists
cursor.execute("DROP TABLE IF EXISTS FactorWeights;")

# --- Create Factor-level Weights table ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS FactorWeights (
    FactorWeightID INTEGER PRIMARY KEY AUTOINCREMENT,
    UserID TEXT NOT NULL,
    IcID INTEGER NOT NULL,
    Factor TEXT NOT NULL,        -- 'P', 'E', 'V'
    WeightValue REAL NOT NULL,
    FOREIGN KEY (UserID) REFERENCES Users(UserID),
    FOREIGN KEY (IcID) REFERENCES ImpactChains(IcID)
);
""")
conn.commit()

# --- Get first UserID ---
cursor.execute("SELECT UserID FROM Users")
user_id = cursor.fetchone()[0]

# --- Load impact chain IDs ---
ic_ids = pd.read_sql_query("SELECT IcID FROM ImpactChains", conn)['IcID'].tolist()

# --- Populate factor-level weights with even weight = 1/3 ---
factors = ['P', 'E', 'V']
equal_weight = 1 / len(factors)  # 0.3333
for ic_id in ic_ids:
    for factor in factors:
        cursor.execute("""
            INSERT INTO FactorWeights (UserID, IcID, Factor, WeightValue)
            VALUES (?, ?, ?, ?)
        """, (user_id, ic_id, factor, equal_weight))

conn.commit()
conn.close()
print("FactorWeights table populated with equal weights (1/3)")


FactorWeights table populated with equal weights (1/3)
